# Preparing Japanese Document for Data generation
- This notebook will show you how to do document parsing
- Document Chunking
- And finally mixing it with user QNA to  create seed examples

## Install SDG

```bash 
pip install sdg-hub[examples]
```

In [ ]:
# import os

# os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [ ]:
force_ascii = True

## Select a document

In [ ]:
use_ibm_annual_report = False
use_teigaku_genzei = True
use_akita_medical = False
use_ibm_newsroom = False
use_ibm_newsroom_en = False
use_jfe_technical_report = False
use_nencho = False

### IBM Annual Report

In [ ]:
if use_ibm_annual_report:
    data_name = "ibm-annual-report"
    data_dir = f"document_collection/{data_name}"
    convert_pdf = True
    docling_config = "docling_v2_config.yaml"
    list_md_files = [f"{data_dir}/ibm-annual-report-2024.md"]
    user_config_path = f"{data_dir}/qna.yaml"

### Teigaku Genzei (by Red Hat Japan)

In [ ]:
if use_teigaku_genzei:
    data_name = "teigaku-genzei"
    data_dir = f"document_collection/{data_name}"
    convert_pdf = True
    docling_config = "docling_v2_config.yaml"
    list_md_files = [f"{data_dir}/0024001-021.md"]
    user_config_path = f"{data_dir}/qna_ja.yaml"

### Akita Medical (by Red Hat Japan)

In [ ]:
if use_akita_medical:
    data_name = "akita-medical"
    data_dir = f"document_collection/{data_name}"
    convert_pdf = True
    docling_config = "docling_v2_config.yaml"
    list_md_files = [f"{data_dir}/funded_medicalcare.md"]
    user_config_path = f"{data_dir}/qna.yaml"

### IBM Newsroom (by IBM Research - Tokyo)

In [ ]:
if use_ibm_newsroom:
    data_name = "ibm-newsroom"
    data_dir = f"document_collection/{data_name}"
    convert_pdf = False
    # docling_config = "docling_v2_config.yaml"
    # list_md_files = [f"{data_dir}/qs2-riken.md", f"{data_dir}/qs1-utokyo.md", f"{data_dir}/jica.md"]
    list_md_files = [f"{data_dir}/qs2-riken.md"]
    user_config_path = f"{data_dir}/qna.yaml"

### IBM Newsroom (English) (by IBM Japan Client Engineering)

In [ ]:
if use_ibm_newsroom_en:
    data_name = "ibm-newsroom-en"
    data_dir = f"document_collection/{data_name}"
    convert_pdf = False
    # docling_config = "docling_v2_config.yaml"
    list_md_files = [f"{data_dir}/IBM_Think_2025.md"]
    user_config_path = f"{data_dir}/qna_Think_2025.yaml"

### JFE Technical Report (by IBM Japan Client Engineering)

In [ ]:
if use_jfe_technical_report:
    data_name = "jfe-technical-report"
    data_dir = f"document_collection/{data_name}"
    convert_pdf = False
    # docling_config = "docling_v2_config.yaml"
    list_md_files = [f"{data_dir}/JFE-InstructLab-TP/Paper{i}.md" for i in range(1, 10)]
    user_config_path = f"{data_dir}/qna.yaml"

### Nencho (WIP) (by IBM Research - Tokyo)

In [ ]:
if use_nencho:
    data_name = "nencho"
    data_dir = f"document_collection/{data_name}"
    convert_pdf = False
    docling_config = "docling_v2_config_no_ocr.yaml"
    list_md_files = [f"{data_dir}/nencho.md"]  # NOTE this file is created by concatenating ??.md after manual cleansing
    user_config_path = f"{data_dir}/qna.yaml"

## Initialize variables

In [ ]:
seed_data_path = f"{data_dir}/seed_data_{data_name}.jsonl"

## Convert PDF to Markdown using docling v2

In [ ]:
if convert_pdf:
    # !OMP_NUM_THREADS=32 mamba run -n docling python docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} --c {docling_config}
    !python docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c {docling_config}

## Create Seed Examples

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.join(os.getcwd())))
from knowledge_utils import DocProcessor

dp = DocProcessor(data_dir, user_config_path=user_config_path)

seed_data = dp.get_processed_markdown_dataset(list_md_files)

seed_data.to_json(seed_data_path, orient='records', lines=True, force_ascii=force_ascii)